In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
# Check GPU is available
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

CUDA available: True
GPU: Tesla T4


### Function to load sample questions from dataset

In [3]:
from datasets import load_dataset
def RAG_Bench_ds_loader(dataset_name, defined_set,testing):
  ds = load_dataset("galileo-ai/ragbench", dataset_name, split=defined_set)
  if testing == True:
    ds = ds[:30]
  df = pd.DataFrame(ds)
  all_docs = []
  for _, row in df.iterrows():
      for doc_pos, doc_text in enumerate(row["documents"]):
          all_docs.append({
              "doc_id":   f"{row['id']}_d{doc_pos}",
              "row_id":   str(row["id"]),
              "doc_pos":  doc_pos,          # 0 to len(documents)-1
              "text":     doc_text.strip(),
              "question": row["question"],
              "response": row["response"]
          })

  docs_df = pd.DataFrame(all_docs)
  print(f"Total documents: {len(docs_df)}")
  return docs_df

### Load Huggingface token

In [4]:
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
groq_token = user_secrets.get_secret("GROQ_TOKEN")

# Set it as an environment variable (what HuggingFace libraries expect)
import os
os.environ["HF_TOKEN"] = hf_token
print("HF_TOKEN set ✓")

HF_TOKEN set ✓


### call the function to load sample

In [5]:
import pandas as pd

docs_df = RAG_Bench_ds_loader("delucionqa","test",True)

README.md: 0.00B [00:00, ?B/s]

delucionqa/train-00000-of-00001.parquet:   0%|          | 0.00/4.23M [00:00<?, ?B/s]

delucionqa/validation-00000-of-00001.par(…):   0%|          | 0.00/528k [00:00<?, ?B/s]

delucionqa/test-00000-of-00001.parquet:   0%|          | 0.00/562k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1460 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/182 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/184 [00:00<?, ? examples/s]

Total documents: 90


### installing required libraries

In [6]:
!pip install -qU \
    "opentelemetry-api>=1.36.0,<1.39.0" \
    "opentelemetry-sdk>=1.36.0,<1.39.0" \
    langchain-core \
    langchain-text-splitters \
    langchain-huggingface \
    langchain-chroma \
    langchain-groq

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.9/554.9 kB 13.6 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 73.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 86.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 89.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.3 MB/s eta 0:00:00


### Phase 1 : Splitting the documents into Chunks

In [11]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,           # characters (not tokens)
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""],  # tries each in order
)


chunks = []
for _, doc in docs_df.iterrows():
    splits = splitter.split_text(doc["text"])
    for chunk_idx, chunk_text in enumerate(splits):
        chunks.append({
            "chunk_id":     f"{doc['doc_id']}_c{chunk_idx}",
            "doc_id":       doc["doc_id"],
            "row_id":       doc["row_id"],
            "doc_pos":      doc["doc_pos"],
            "chunk_idx":    chunk_idx,
            "total_chunks": len(splits),
            "text":         chunk_text,
        })


### Converting them to Langchain Documents

In [12]:
from langchain_core.documents import Document
from langchain_chroma import Chroma

# Convert dicts → Document objects
# 'text' becomes page_content, everything else goes into metadata
documents = [
    Document(
        page_content=chunk["text"],
        metadata={
            "chunk_id"    : chunk["chunk_id"],
            "doc_id"      : chunk["doc_id"],
            "row_id"      : chunk["row_id"],
            "doc_pos"     : chunk["doc_pos"],
        }
    )
    for chunk in chunks
]

print(f"Converted {len(documents)} dicts → Document objects")
print(f"Sample page_content : {documents[0].page_content[:100]}")
print(f"Sample metadata     : {documents[0].metadata}")

Converted 304 dicts → Document objects
Sample page_content : Closing To close the tailgate, lift upward until both sides latch into place.  CAUTION: After closin
Sample metadata     : {'chunk_id': '114_d0_c0', 'doc_id': '114_d0', 'row_id': '114', 'doc_pos': 0}


### Loading the and configuring the embedding model

In [13]:
from langchain_huggingface import HuggingFaceEmbeddings

# DO NOT specify device—let it auto-detect GPU
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5",
    encode_kwargs={"normalize_embeddings": True},
)

print(f"Embedding model loaded (auto-detected device)")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded (auto-detected device)


### Creating Vector DB with index and store the embeddings

In [14]:
CHROMA_PATH = "/kaggle/working/chroma_db"

# Build index here, then download it from the output panel
vector_store = Chroma.from_documents(
    documents=documents,
    embedding=embedding_model,
    collection_name="ragbench_delucion",
    persist_directory=CHROMA_PATH,
)

### RAG Pipeline

In [15]:
import os
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# ── LLM (replaces groq.Groq client) ─────────────────────────────────────────
llm = ChatGroq(
    api_key=groq_token,
    model="llama-3.1-8b-instant",
    temperature=0.0,
    max_tokens=512,
)

# ── Prompt (replaces your f-string) ─────────────────────────────────────────
prompt = ChatPromptTemplate.from_messages([
    ("system", """Answer the question based only on the context. 
Use the complete context and infer the answer, don't conclude individually.

Context:
{context}"""),
    ("human", "{question}"),
])

# ── Retriever (replaces your retrieve() function) ───────────────────────────
retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 4, "fetch_k": 20, "lambda_mult": 0.7},
)

# ── format_docs (replaces your build_context()) ─────────────────────────────
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# ── LCEL chain (replaces your generate() function) ──────────────────────────
rag_chain = (
    {
        "context" : retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [16]:
question = docs_df["question"].iloc[19]

answer = rag_chain.invoke(question)

print(f"Question     : {question}")
print(f"\nGenerated    :\n{answer}")
print(f"\nGround Truth :\n{docs_df['response'].iloc[19]}")

Question     : how to calculate the gross trailer weight?

Generated    :
Based on the context, the gross trailer weight (GTW) is the weight of the trailer plus the weight of all cargo, consumables, and equipment (permanent or temporary) loaded in or on the trailer in its "loaded and ready for operation" condition. 

The recommended way to measure GTW is to put your fully loaded trailer on a vehicle scale. The entire weight of the trailer must be supported by the scale.

Ground Truth :
To calculate the gross trailer weight (GTW), the recommended way is to put your fully loaded trailer on a vehicle scale where the entire weight of the trailer must be supported by the scale. The GTW is the weight of the trailer plus the weight of all cargo, consumables, and equipment (permanent or temporary) loaded in or on the trailer in its "loaded and ready for operation" condition. This measurement should be taken when the trailer is fully loaded and ready to be towed.


### Evaluators

### a) Using LLM as Judge

In [12]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import json, re
from nltk.tokenize import sent_tokenize
import nltk
nltk.download('punkt_tab', quiet=True)

True

In [13]:
def get_sentences(text: str) -> list[str]:
    """Split text into sentences, return as indexed list."""
    return [s.strip() for s in sent_tokenize(text) if s.strip()]

In [14]:
# ── LLM Span Identifier (fixed prompts) ──────────────────────────────────────

span_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a precise text analysis assistant.
You will be given numbered sentences and a task.
Respond ONLY with a JSON array of sentence numbers.
Example: [1, 3, 5]
If none qualify, respond with: []
No explanation. No other text."""),
    ("human", "{task}\n\nSentences:\n{numbered_sentences}"),
])

span_chain = span_prompt | llm | StrOutputParser()


def get_relevant_indices(sentences: list[str], query: str) -> list[int]:
    """Ri — sentences that contain information useful for answering the query."""
    if not sentences:
        return []

    numbered = "\n".join(f"{i+1}. {s}" for i, s in enumerate(sentences))
    task = (
        f"Which sentences contain information that helps answer this query?\n"
        f"Query: {query}\n"
        f"Select ALL sentences that are topically relevant, even partially."
    )
    raw = span_chain.invoke({"task": task, "numbered_sentences": numbered})
    try:
        match = re.search(r'\[.*?\]', raw, re.DOTALL)
        indices = json.loads(match.group()) if match else []
        return [i - 1 for i in indices if 1 <= i <= len(sentences)]
    except Exception:
        return []


def get_utilized_indices(sentences: list[str], response: str) -> list[int]:
    """
    Ui — sentences whose INFORMATION (not exact words) was used
    by the generator to produce the response.
    """
    if not sentences:
        return []

    numbered = "\n".join(f"{i+1}. {s}" for i, s in enumerate(sentences))
    task = (
        f"Which sentences provided the information or facts present in this response?\n"
        f"Response: {response}\n"
        f"A sentence qualifies even if the response paraphrases it, not quotes it verbatim."
    )
    raw = span_chain.invoke({"task": task, "numbered_sentences": numbered})
    try:
        match = re.search(r'\[.*?\]', raw, re.DOTALL)
        indices = json.loads(match.group()) if match else []
        return [i - 1 for i in indices if 1 <= i <= len(sentences)]
    except Exception:
        return []


def get_grounded_indices(response_sentences: list[str], context: str) -> list[int]:
    """
    Adherence — which response sentences are supported by the context,
    including paraphrases and inferences directly derivable from it.
    """
    if not response_sentences:
        return []

    numbered = "\n".join(f"{i+1}. {s}" for i, s in enumerate(response_sentences))
    task = (
        f"Which response sentences are supported by information in this context?\n"
        f"A sentence is supported even if it paraphrases the context.\n"
        f"Context: {context}"
    )
    raw = span_chain.invoke({"task": task, "numbered_sentences": numbered})
    try:
        match = re.search(r'\[.*?\]', raw, re.DOTALL)
        indices = json.loads(match.group()) if match else []
        return [i - 1 for i in indices if 1 <= i <= len(response_sentences)]
    except Exception:
        return []

In [15]:
# ── TRACe Metric Functions (LLM-based, paper formulas) ───────────────────────

def context_relevance(contexts: list[str], query: str) -> float:
    """
    Eq. 2 — Σ Len(Ri) / Σ Len(di)
    LLM identifies which context sentences are relevant to the query.
    """
    total_len    = 0
    relevant_len = 0

    for doc in contexts:
        sentences    = get_sentences(doc)
        relevant_idx = get_relevant_indices(sentences, query)
        total_len   += len(sentences)
        relevant_len += len(relevant_idx)

    return relevant_len / total_len if total_len > 0 else 0.0


def context_utilization(contexts: list[str], response: str) -> float:
    """
    Eq. 3 — Σ Len(Ui) / Σ Len(di)
    LLM identifies which context sentences provided information
    used in the response (including paraphrases).
    """
    total_len    = 0
    utilized_len = 0

    for doc in contexts:
        sentences    = get_sentences(doc)
        utilized_idx = get_utilized_indices(sentences, response)
        total_len   += len(sentences)
        utilized_len += len(utilized_idx)

    return utilized_len / total_len if total_len > 0 else 0.0


def completeness(contexts: list[str], query: str, response: str) -> float:
    """
    Eq. 4 — Σ Len(Ri ∩ Ui) / Σ Len(Ri)
    Fraction of relevant sentences that were also utilized.
    Caches relevant/utilized indices per doc to avoid duplicate LLM calls.
    """
    total_relevant        = 0
    relevant_and_utilized = 0

    for doc in contexts:
        sentences    = get_sentences(doc)
        relevant_idx = set(get_relevant_indices(sentences, query))
        utilized_idx = set(get_utilized_indices(sentences, response))

        # Ri ∩ Ui — sentences that are both relevant AND utilized
        intersection           = relevant_idx & utilized_idx
        total_relevant        += len(relevant_idx)
        relevant_and_utilized += len(intersection)

    return relevant_and_utilized / total_relevant if total_relevant > 0 else 0.0


def adherence(contexts: list[str], response: str) -> float:
    """
    Fraction of response sentences grounded in context.
    Paper defines this as boolean (all or nothing) at example level,
    but we return a float [0, 1] for richer evaluation signal.
    Grounding check includes paraphrases, not just verbatim matches.
    """
    response_sentences = get_sentences(response)
    if not response_sentences:
        return 0.0

    full_context = "\n\n".join(contexts)
    grounded_idx = get_grounded_indices(response_sentences, full_context)

    return len(grounded_idx) / len(response_sentences)

In [16]:
def compute_trace_metrics(
    query    : str,
    contexts : list[str],
    response : str,
) -> dict:
    """
    Compute all four TRACe metrics using LLM-based span identification.

    Args:
        query    : user question
        contexts : list of retrieved context strings (one per chunk)
        response : generated answer
    Returns:
        dict with keys: context_relevance, context_utilization,
                        completeness, adherence
    """
    return {
        "context_relevance"  : context_relevance(contexts, query),
        "context_utilization": context_utilization(contexts, response),
        "completeness"       : completeness(contexts, query, response),
        "adherence"          : adherence(contexts, response),
    }

### b) Using Base NLI DeBERTa (works today)

In [17]:
!pip install -q transformers torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 97.4 MB/s eta 0:00:00:00:01:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which 

In [18]:
from transformers import pipeline
from nltk.tokenize import sent_tokenize
import nltk
nltk.download('punkt_tab', quiet=True)

# The exact base model the paper fine-tuned from
nli_model = pipeline(
    "zero-shot-classification",
    model="MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli",
    device=0,   # GPU; use -1 for CPU
)

print("NLI model loaded ✓")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/870M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/395 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/18.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

NLI model loaded ✓


In [19]:
def is_entailed(premise: str, hypothesis: str, threshold: float = 0.5) -> bool:
    """
    Returns True if hypothesis is entailed by premise.
    Uses NLI entailment score as a proxy for span support.
    """
    result = nli_model(
        sequences=premise,
        candidate_labels=[hypothesis],
        hypothesis_template="{}",
    )
    return result["scores"][0] >= threshold


def get_sentences(text: str) -> list[str]:
    return [s.strip() for s in sent_tokenize(text) if s.strip()]

In [20]:
def context_relevance(contexts: list[str], query: str) -> float:
    """Eq. 2 — fraction of context sentences entailed by / relevant to query."""
    total = 0
    relevant = 0
    for doc in contexts:
        sentences = get_sentences(doc)
        for sent in sentences:
            total += 1
            if is_entailed(premise=sent, hypothesis=query):
                relevant += 1
    return relevant / total if total > 0 else 0.0


def context_utilization(contexts: list[str], response: str) -> float:
    """Eq. 3 — fraction of context sentences entailed by the response."""
    total = 0
    utilized = 0
    for doc in contexts:
        sentences = get_sentences(doc)
        for sent in sentences:
            total += 1
            if is_entailed(premise=sent, hypothesis=response):
                utilized += 1
    return utilized / total if total > 0 else 0.0


def completeness(contexts: list[str], query: str, response: str) -> float:
    """Eq. 4 — of relevant sentences, how many were also utilized."""
    total_relevant = 0
    relevant_and_utilized = 0
    for doc in contexts:
        sentences = get_sentences(doc)
        for sent in sentences:
            rel  = is_entailed(premise=sent, hypothesis=query)
            util = is_entailed(premise=sent, hypothesis=response)
            if rel:
                total_relevant += 1
                if util:
                    relevant_and_utilized += 1
    return relevant_and_utilized / total_relevant if total_relevant > 0 else 0.0


def adherence(contexts: list[str], response: str) -> float:
    """Fraction of response sentences entailed by the context."""
    full_context = " ".join(contexts)
    response_sentences = get_sentences(response)
    if not response_sentences:
        return 0.0
    grounded = sum(
        1 for sent in response_sentences
        if is_entailed(premise=full_context, hypothesis=sent)
    )
    return grounded / len(response_sentences)


def compute_trace_metrics(query: str, contexts: list[str], response: str) -> dict:
    return {
        "context_relevance"  : context_relevance(contexts, query),
        "context_utilization": context_utilization(contexts, response),
        "completeness"       : completeness(contexts, query, response),
        "adherence"          : adherence(contexts, response),
    }

In [21]:
question       = docs_df["question"].iloc[29]
retrieved_docs = retriever.invoke(question)
contexts       = [doc.page_content for doc in retrieved_docs]
response       = rag_chain.invoke(question)

scores = compute_trace_metrics(
    query    = question,
    contexts = contexts,
    response = response,
)

print(f"Question     : {question}")
print(f"\nGenerated    : {response}")
print(f"\nContext : {contexts}")
print(f"\nGround Truth : {docs_df['response'].iloc[29]}")
print(f"\n── TRACe Scores ──────────────────")
for metric, score in scores.items():
    print(f"  {metric:<22}: {score:.4f}")

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Question     : Can I manually activate or deactivate the air conditioner?

Generated    : Yes, you can manually activate or deactivate the air conditioner using the A/C button.

Context : ['The Air Conditioning (A/C) button allows the operator to manually activate or deactivate the air conditioning system.  When the air conditioning system is turned on, cool dehumidified air will flow through the outlets into the cabin.  If the light turns on while driving, safely pull over and stop the vehicle.  If the Air Conditioning (A/C) system is on, turn it off.  Also, shift the transmission into NEUTRAL and idle the vehicle', 'A/C Button Press and release this button on the touchscreen, or push the button on the faceplate to change the current setting.  The A/C indicator illuminates when A/C is ON.  The Air Conditioning (A/C) button allows the operator to manually activate or deactivate the air conditioning system.  When the air conditioning system is turned on, cool dehumidified air will flow 

In [26]:
# Get unique questions from your already-loaded docs_df
# group by row_id to get one entry per original RAGBench example
questions_df = (
    docs_df
    .groupby("row_id", sort=False)
    .agg(
        question=("question", "first"),
        response=("response",  "first"),
    )
    .reset_index()
)

print(f"Unique questions: {len(questions_df)}")
print(questions_df[["row_id", "question"]].head())

Unique questions: 30
  row_id                                           question
0    114     What if I fail to latch the tailgate properly?
1    380  What kind of safety features are implemented i...
2    826          When will the Automatic SOS be triggered?
3    718  What happens if I accidentally push the SOS Ca...
4     31                                   What is the DEF?


In [30]:
results = []
import random


for i in range(15):
    row          = questions_df.iloc[i]
    question     = row["question"]
    ground_truth = row["response"]

    # Retrieve from YOUR Chroma vector store
    retrieved_docs = retriever.invoke(question)
    contexts       = [doc.page_content for doc in retrieved_docs]

    # Generate with YOUR RAG chain
    response = rag_chain.invoke(question)

    # TRACe scores
    scores = compute_trace_metrics(question, contexts, response)

    results.append({
        "idx"         : i,
        "question"    : question,
        "generated"   : response,
        "ground_truth": ground_truth,
        **scores,
    })

    print(f"✓ [{i+1}/5] {question[:60]}...")

print("\nDone.")

✓ [1/5] What if I fail to latch the tailgate properly?...
✓ [2/5] What kind of safety features are implemented in this car?...
✓ [3/5] When will the Automatic SOS be triggered?...
✓ [4/5] What happens if I accidentally push the SOS Call button?...
✓ [5/5] What is the DEF?...
✓ [6/5] What may cause erratic or noisy performance of the radio?...
✓ [7/5] how to calculate the gross trailer weight?...
✓ [8/5] What can the ASIST button do?...
✓ [9/5] What does the Door Off Mirror Kit do?...
✓ [10/5] Can I manually activate or deactivate the air conditioner?...
✓ [11/5] what does the The cartridge block heater do?...
✓ [12/5] When will the illuminated entry system be activated?...
✓ [13/5] If the light remains on after the bulb check, it indicates w...
✓ [14/5] What is special about power windows?...
✓ [15/5] What should I do to jump start the car?...

Done.


In [31]:
results_df = pd.DataFrame(results)

print("── Per-Example TRACe Scores ─────────────────────────────────")
cols = ["idx", "question", "context_relevance",
        "context_utilization", "completeness", "adherence"]

display(
    results_df[cols]
    .style
    .format({
        "context_relevance"  : "{:.3f}",
        "context_utilization": "{:.3f}",
        "completeness"       : "{:.3f}",
        "adherence"          : "{:.3f}",
    })
    .background_gradient(
        subset=["context_relevance","context_utilization",
                "completeness","adherence"],
        cmap="RdYlGn", vmin=0, vmax=1
    )
)

print("\n── Aggregate TRACe Scores (mean over 5 examples) ────────────")
metric_cols = ["context_relevance","context_utilization",
               "completeness","adherence"]

for metric in metric_cols:
    score = results_df[metric].mean()
    bar   = "█" * int(score * 20)
    space = "░" * (20 - len(bar))
    print(f"  {metric:<22}: {score:.4f}  |{bar}{space}|")

── Per-Example TRACe Scores ─────────────────────────────────


,idx,question,context_relevance,context_utilization,completeness,adherence
0,0,What if I fail to latch the tailgate properly?,0.579,0.947,1.000,1.000
1,1,What kind of safety features are implemented in this car?,0.588,1.000,1.000,1.000
2,2,When will the Automatic SOS be triggered?,0.231,0.077,0.333,1.000
3,3,What happens if I accidentally push the SOS Call button?,0.692,0.615,0.778,1.000
4,4,What is the DEF?,0.143,1.000,1.000,1.000
5,5,What may cause erratic or noisy performance of the radio?,0.500,0.929,1.000,1.000
6,6,how to calculate the gross trailer weight?,0.429,1.000,1.000,1.000
7,7,What can the ASIST button do?,0.400,0.333,0.333,1.000
8,8,What does the Door Off Mirror Kit do?,0.333,0.500,1.000,1.000
9,9,Can I manually activate or deactivate the air conditioner?,0.526,0.789,1.000,1.000



── Aggregate TRACe Scores (mean over 5 examples) ────────────
  context_relevance     : 0.4472  |████████░░░░░░░░░░░░|
  context_utilization   : 0.7195  |██████████████░░░░░░|
  completeness          : 0.8106  |████████████████░░░░|
  adherence             : 0.9667  |███████████████████░|
